# Daily Challenge: How to Fine-tune LLMs with LoRA

Week 13, Day 2

**Parameter-Efficient Fine-Tuning (PEFT)** methods like **LoRA** adapt a large pre-trained model by training only a small number of extra parameters (low-rank matrices added to some layers) while the original weights stay frozen. This reduces memory, compute and storage: the adapter is a few MB instead of a full copy of the model.

**Goal:** fine-tune `bigscience/bloomz-560m` with LoRA on 10% of the `Abirate/english_quotes` dataset so that it generates quotes, then save the adapter, reload it with `PeftModel.from_pretrained` and generate text.

## 1. Install the necessary libraries

> The hint pins `peft==0.4.0` (2023). That version is no longer compatible with recent versions of `transformers` (including the one pre-installed in Colab), so we install a recent `peft`. The API used below (`LoraConfig`, `get_peft_model`, `PeftModel.from_pretrained`) is the same.

In [ ]:
%pip install -q peft datasets accelerate
!mkdir -p cache

In [ ]:
import os, time, warnings
os.environ["USE_TF"] = "0"              # PyTorch backend only
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
import transformers
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
import peft, datasets
transformers.set_seed(42)
print("transformers", transformers.__version__, "| peft", peft.__version__, "| datasets", datasets.__version__,
      "| GPU:", torch.cuda.is_available())

## 2. Load the pre-trained model and its tokenizer

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)
print(f"{model_name}: {foundation_model.num_parameters():,} parameters")
print(foundation_model.transformer.h[0])   # one transformer block: the attention uses a fused 'query_key_value' linear layer

BLOOMZ-560M is a 560-million-parameter decoder-only (GPT-like) model from BigScience, multitask-fine-tuned to follow prompts. Each of its 24 blocks has a self-attention layer where the query, key and value projections are fused into a single linear layer called **`query_key_value`**: this is the layer we will adapt with LoRA.

Before fine-tuning, let's look at what the original model generates for our prompt:

In [ ]:
def generate(model, prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        repetition_penalty=1.5,          # avoid repeating the same words
        eos_token_id=tokenizer.eos_token_id,
        do_sample=False,                 # deterministic (greedy) decoding to compare models fairly
    )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

prompts = ["Two things are infinite: ", "Be yourself; ", "Life is like "]
baseline_outputs = {p: generate(foundation_model, p)[0] for p in prompts}
for p, out in baseline_outputs.items():
    print(f"[original model] {out!r}")

## 3. Load and preprocess the dataset

In [ ]:
data = load_dataset("Abirate/english_quotes", split="train[:10%]")  # Sample 10%
print(data)
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)
train_sample = data.select(range(5))
display(train_sample)
for row in train_sample:
    print(f"{row['quote']}  — {row['author']}  ({len(row['input_ids'])} tokens)")

The dataset contains famous quotes with their author and tags; the 10% sample has **251 quotes**. We only use the `quote` column: each quote is tokenized into `input_ids` and an `attention_mask`, the format expected by the model. The language-modeling collator used later will copy `input_ids` into `labels`, so the model learns to predict each next token of the quotes (causal language modeling), i.e. to write in the style of these quotes.

## 4–5. Configure LoRA and apply it to the model

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,                                  # rank of the low-rank update matrices
    lora_alpha=1,                         # a scaling factor that adjusts the magnitude of the weight matrix. Usually set to 1
    target_modules=["query_key_value"],   # the fused attention projection of every BLOOM block
    lora_dropout=0.05,                    # dropout on the LoRA branch to limit overfitting
    bias="none",                          # this specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM"
)

# Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())
print(peft_model.base_model.model.transformer.h[0].self_attention.query_key_value)

With **rank r = 1** on the `query_key_value` layers (1024 → 3072) of the 24 blocks, LoRA adds only `24 × 1 × (1024 + 3072) = 98,304` trainable parameters: **about 0.02% of the 559 million parameters**, which all stay frozen. `print_trainable_parameters()` prints this summary (and returns `None`, hence the extra `None` line). The printed layer shows that each `query_key_value` is now a `lora.Linear` wrapping the original layer with two small matrices `lora_A` (1024 → 1) and `lora_B` (1 → 3072).

## 6–7. Set up the training arguments and train with `Trainer`

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,        # find the largest batch size that fits in memory
    learning_rate=3e-2,               # Higher learning rate than full fine-tuning.
    num_train_epochs=2,
    logging_steps=4,
    save_strategy="no",
    use_cpu=not torch.cuda.is_available(),
    seed=42,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
t0 = time.time()
trainer.train()
print(f"Training time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
logs = pd.DataFrame([l for l in trainer.state.log_history if 'loss' in l])
display(logs[['step', 'epoch', 'loss']])
logs.plot(x='step', y='loss', marker='o', figsize=(7, 3), title='Training loss (LoRA fine-tuning)', legend=False)
plt.ylabel('loss'); plt.show()

## 8. Save the fine-tuned LoRA model

In [ ]:
import time

time_now = time.strftime("%Y%m%d-%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

print("Saved to:", peft_model_path)
for f in sorted(os.listdir(peft_model_path)):
    print(f"  {f}: {os.path.getsize(os.path.join(peft_model_path, f)) / 1024:.1f} KB")

Only the **adapter** is saved: `adapter_config.json` (the LoRA configuration and the name of the base model) and `adapter_model.safetensors` with the LoRA weights, a few hundred KB, compared with about 1.1 GB for the full BLOOMZ-560M model. To use it, we load the base model and attach the adapter.

## 9. Load the saved LoRA model for inference

In [ ]:
from peft import PeftModel

# Load the PEFT model using pre-defined LoRA configs and foundation model. We set `is_trainable=False` to avoid further training.
base_model = AutoModelForCausalLM.from_pretrained(model_name)
loaded_model = PeftModel.from_pretrained(base_model, peft_model_path, is_trainable=False)
loaded_model.eval()
loaded_model.print_trainable_parameters()   # 0 trainable parameters: inference only

## 10. Generate text with the fine-tuned model

In [ ]:
# Generate output tokens
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=20,
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
    do_sample=False,
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

In [ ]:
# Before vs after fine-tuning on several prompts
for p in prompts:
    print(f"Prompt: {p!r}")
    print(f"  original model : {baseline_outputs[p]!r}")
    print(f"  LoRA fine-tuned: {generate(loaded_model, p)[0]!r}\n")

RESULTS_PLACEHOLDER